DATASET 5,066장을 Train 70% (4,172장), Val 15% (894장)로 재분할

📁 결과물 위치
입력:

N:\개인\이수빈\3.13_Mini_Project\DATASET\fire_5000\

출력:

N:\개인\이수빈\3.13_Mini_Project\DATASET\fire_split_70_15\

train/images/ - 4,172장

train/labels/ - 4,172개

val/images/ - 894장

val/labels/ - 894개

data_70_15.yaml

In [2]:
from pathlib import Path  # 경로 처리 라이브러리
import shutil  # 파일 복사 라이브러리
import random  # 랜덤 셔플용

# ============================================
# 5,066장을 70:15 비율로 재분할
# (Train 4,172장 + Val 894장)
# ============================================

# 프로젝트 루트 경로 설정
PROJECT_ROOT = Path(r'N:\개인\이수빈\3.13_Mini_Project')

# 원본 데이터 경로 (기존 fire_5000 폴더)
ORIGINAL_DIR = PROJECT_ROOT / 'DATASET' / 'fire_5000'

# 새로 분할된 데이터를 저장할 폴더 경로
SPLIT_DIR = PROJECT_ROOT / 'DATASET' / '5000_split_70_15'
# Train 폴더 경로
TRAIN_DIR = SPLIT_DIR / 'train'
# Validation 폴더 경로
VAL_DIR = SPLIT_DIR / 'val'

# Train, Val 폴더 생성 (images, labels 서브폴더 포함)
for folder in [TRAIN_DIR / 'images', TRAIN_DIR / 'labels',  # Train용 폴더
               VAL_DIR / 'images', VAL_DIR / 'labels']:      # Val용 폴더
    folder.mkdir(parents=True, exist_ok=True)  # 폴더 생성 (이미 있으면 무시)

# 진행 상황 헤더 출력
print("=" * 60)
print("📂 데이터 재분할 (70:15)")
print("=" * 60)

# ============================================
# 1. 전체 이미지 파일 리스트 가져오기
# ============================================
# fire_5000/images 폴더의 모든 .jpg 파일 찾기
all_images = list((ORIGINAL_DIR / 'images').glob('*.jpg'))

# 전체 이미지 개수 출력
print(f"\n전체 이미지: {len(all_images):,}장")

# ============================================
# 2. 랜덤 섞기 (재현 가능하도록 시드 고정)
# ============================================
random.seed(42)  # 시드 고정 (같은 결과 재현 가능)
random.shuffle(all_images)  # 리스트 랜덤 섞기

# ============================================
# 3. 70:15 비율로 분할
# ============================================
# 전체 이미지 개수
total = len(all_images)

# Train 개수 계산 (전체의 70/85 = 82.35%)
# 85%는 Train+Val이므로, 70%는 전체의 70/85
train_count = int(total * 0.70 / 0.85)  # 4,172장

# Val 개수는 나머지
val_count = total - train_count  # 894장

# 리스트 슬라이싱으로 분할
train_images = all_images[:train_count]  # 처음부터 4,172장까지 = Train
val_images = all_images[train_count:]    # 나머지 894장 = Val

# 분할 결과 출력
print(f"\n분할:")
print(f"  Train: {len(train_images):,}장 ({len(train_images)/total*100:.1f}%)")
print(f"  Val:   {len(val_images):,}장 ({len(val_images)/total*100:.1f}%)")

# ============================================
# 4. Train 데이터 복사
# ============================================
print(f"\n[Train 복사 중...]")

# Train 이미지 각각에 대해 반복
for idx, img_path in enumerate(train_images, 1):  # enumerate로 인덱스와 함께 순회
    # 이미지 복사 경로 생성
    dst_img = TRAIN_DIR / 'images' / img_path.name
    # 이미지 파일 복사 (원본 유지)
    shutil.copy2(img_path, dst_img)
    
    # 해당 이미지의 라벨 파일 경로 (확장자만 .txt로)
    label_path = ORIGINAL_DIR / 'labels' / f"{img_path.stem}.txt"
    # 라벨 복사 경로 생성
    dst_label = TRAIN_DIR / 'labels' / f"{img_path.stem}.txt"
    # 라벨 파일 복사
    shutil.copy2(label_path, dst_label)
    
    # 진행 상황 출력 (500장마다)
    if idx % 500 == 0:
        print(f"  {idx}/{len(train_images)}")

# Train 복사 완료 메시지
print(f"  ✅ Train {len(train_images):,}장 완료")

# ============================================
# 5. Val 데이터 복사
# ============================================
print(f"\n[Val 복사 중...]")

# Val 이미지 각각에 대해 반복
for idx, img_path in enumerate(val_images, 1):  # enumerate로 인덱스와 함께 순회
    # 이미지 복사 경로 생성
    dst_img = VAL_DIR / 'images' / img_path.name
    # 이미지 파일 복사
    shutil.copy2(img_path, dst_img)
    
    # 해당 이미지의 라벨 파일 경로
    label_path = ORIGINAL_DIR / 'labels' / f"{img_path.stem}.txt"
    # 라벨 복사 경로 생성
    dst_label = VAL_DIR / 'labels' / f"{img_path.stem}.txt"
    # 라벨 파일 복사
    shutil.copy2(label_path, dst_label)
    
    # 진행 상황 출력 (100장마다)
    if idx % 100 == 0:
        print(f"  {idx}/{len(val_images)}")

# Val 복사 완료 메시지
print(f"  ✅ Val {len(val_images):,}장 완료")

# ============================================
# 6. YOLO 학습용 data.yaml 파일 생성
# ============================================
# yaml 파일 내용 구성 (경로, 클래스 정보)
yaml_content = f"""path: {str(SPLIT_DIR).replace(chr(92), '/')}
train: train/images
val: val/images

nc: 2
names: ['fire', 'smoke']
"""

# yaml 파일 저장 경로
yaml_path = SPLIT_DIR / 'data_70_15.yaml'

# yaml 파일 쓰기
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

# yaml 파일 생성 완료 메시지
print(f"\n✅ data.yaml 생성: {yaml_path}")

# ============================================
# 7. 최종 확인 (실제 복사된 파일 개수 확인)
# ============================================
print(f"\n{'='*60}")
print("📊 재분할 완료")
print(f"{'='*60}")

# Train 폴더의 실제 이미지 개수 확인
train_imgs = list((TRAIN_DIR / 'images').glob('*.jpg'))
# Val 폴더의 실제 이미지 개수 확인
val_imgs = list((VAL_DIR / 'images').glob('*.jpg'))

# 저장 위치 출력
print(f"\n저장 위치: {SPLIT_DIR}")

# 실제 복사된 개수 출력
print(f"\nTrain: {len(train_imgs):,}장")
print(f"Val:   {len(val_imgs):,}장")
print(f"총:    {len(train_imgs) + len(val_imgs):,}장")

# 실제 비율 계산 및 출력
total_copied = len(train_imgs) + len(val_imgs)
print(f"\n비율:")
print(f"  Train: {len(train_imgs)/total_copied*100:.1f}%")
print(f"  Val:   {len(val_imgs)/total_copied*100:.1f}%")

# 다음 단계 안내
print(f"\n다음: 이 데이터로 재학습")
print(f"{'='*60}")

📂 데이터 재분할 (70:15)

전체 이미지: 5,066장

분할:
  Train: 4,172장 (82.4%)
  Val:   894장 (17.6%)

[Train 복사 중...]
  500/4172
  1000/4172
  1500/4172
  2000/4172
  2500/4172
  3000/4172
  3500/4172
  4000/4172
  ✅ Train 4,172장 완료

[Val 복사 중...]
  100/894
  200/894
  300/894
  400/894
  500/894
  600/894
  700/894
  800/894
  ✅ Val 894장 완료

✅ data.yaml 생성: N:\개인\이수빈\3.13_Mini_Project\DATASET\5000_split_70_15\data_70_15.yaml

📊 재분할 완료

저장 위치: N:\개인\이수빈\3.13_Mini_Project\DATASET\5000_split_70_15

Train: 4,172장
Val:   894장
총:    5,066장

비율:
  Train: 82.4%
  Val:   17.6%

다음: 이 데이터로 재학습
